# H, answer-conditioned plan search: Stage 0, the specificity test

This is evaluation only and uses training questions only. Every criterion is frozen in
`results/plan_search/stage0/PLAN.md`, and the notebook refuses to start unless it matches `PLAN.sha256`.

1. **Phase 1, at most 8 minutes:** greedy on GSM8K-Aug training questions until 1,000 are wrong. Each wrong question gets
   8 dropout samples of z₁, whose spread sets its radius.
2. **Phase 2, at most 12 minutes:** for each wrong question, 20 gradient steps on z₁, kept within the radius and inside
   the top 64 principal directions. One search aims at the gold answer, and one at a wrong answer the model finds just as
   likely.
   - A plan is accepted if the model's own latent steps from it then give exactly that answer.
3. **Gate:** **GO** if gold plans are accepted at least 1.5× as often as wrong ones, with at least 150 gold plans.
   Otherwise **KILL**: the search would be writing answers into z₁ rather than finding plans.

**Before running:** Runtime → Change runtime type → **G4**, then Runtime → **Run all**. If Colab disconnects, Run all again
resumes the search; one resume is allowed.

In [ ]:
REPO_REF = "main"         # the branch or commit holding scripts/plan_search and the frozen plan

In [ ]:
from pathlib import Path
import hashlib, json, os, runpy, shutil, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/dynamic-state-anchoring")
BACKUP_DIR = DRIVE / "plan-search" / "stage0"
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode:
    raise SystemExit("No GPU: Runtime > Change runtime type > GPU, then Runtime > Run all.")
print("GPU:", gpu.stdout.strip())

PINNED = BACKUP_DIR / "commit.txt"  # a started run keeps the code it started with
ref = PINNED.read_text().strip() if PINNED.exists() else REPO_REF
REPO_DIR = Path("/content/dynamic-state-anchoring")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/sttruji/dynamic-state-anchoring.git", str(REPO_DIR)],
                   check=True)
subprocess.run(["git", "fetch", "origin", ref], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
plan = Path("results/plan_search/stage0/PLAN.md")
pinned = Path("results/plan_search/stage0/PLAN.sha256").read_text().split()[0]
if hashlib.sha256(plan.read_bytes()).hexdigest() != pinned:
    raise SystemExit("PLAN.md does not match PLAN.sha256: the frozen plan was changed; stop.")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
PINNED.write_text(COMMIT)
sessions = BACKUP_DIR / "sessions.txt"
count = int(sessions.read_text()) + 1 if sessions.exists() else 1
if count > 2:
    raise SystemExit("Stage 0 allows one session and one resume; this would be a third.")
sessions.write_text(str(count))
print("Code:", COMMIT, f"(from {ref}); plan sha256 matches; session {count} of at most 2")

In [ ]:
VENV = Path("/content/dsa-venv")
subprocess.run([sys.executable, "scripts/setup_colab.py", "--venv-dir", str(VENV),
                "--requirements", "requirements-colab.txt"], check=True)
PYTHON = str(VENV / "bin/python")
os.environ["DSA_DATA_DIR"] = str(DRIVE / "data")
os.environ["HF_HOME"] = "/content/hf-cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
run_logged = runpy.run_path("scripts/colab_process.py")["run_logged"]

def run(*args):
    run_logged([PYTHON, *map(str, args)], cwd=REPO_DIR, log_dir=BACKUP_DIR / "logs")

TRAIN = DRIVE / "data" / "gsm_train.json"  # the GSM8K-Aug training file the inner-loop runs used
assert TRAIN.exists(), f"{TRAIN} is missing; it is the training file from the inner-loop notebook's runs"
CHECKPOINT = DRIVE / "checkpoints/simcot-gpt2"
run("scripts/latent_gate/fetch_simcot.py", "--output", CHECKPOINT)

In [ ]:
RUN = BACKUP_DIR / "run"
run("scripts/plan_search/stage0.py", "--checkpoint", CHECKPOINT, "--output", RUN, "--train-file", TRAIN,
    "--device", "cuda")
print(json.dumps(json.loads((RUN / "status.json").read_text()), indent=1))

In [ ]:
report = subprocess.run([PYTHON, "scripts/plan_search/report.py", str(RUN)], cwd=REPO_DIR,
                        capture_output=True, text=True)
print(report.stdout, report.stderr)
(BACKUP_DIR / "report.txt").write_text(report.stdout)

In [ ]:
# The records, status and report, for the repo's results folder.
from google.colab import files

bundle = Path("/content/plan-search-stage0")
shutil.rmtree(bundle, ignore_errors=True)
shutil.copytree(RUN, bundle / "run")
for name in ("report.txt", "commit.txt", "sessions.txt"):
    shutil.copy2(BACKUP_DIR / name, bundle / name)
files.download(shutil.make_archive(str(bundle), "zip", bundle))
print(f"Downloading {bundle}.zip. If it does not start, use the Files panel on the left.")

**Reading the result**
- **Phase 1** gives greedy accuracy on the training questions, which should be high (about 78%), and the identity check,
  which must pass on 20 questions.
- **Phase 2** gives gold and wrong acceptance, how many questions accept both, how often plans hit the radius, and
  log-probability gains.
- **The last line** is GO or KILL by the frozen gate.

**Sharing:** the last cell downloads a zip. Point Claude at it.